In [1]:
import os
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm

import mediapipe as mp

In [2]:
IMG_ROOT = Path("/kaggle/input/datasets/hooriyamasood80/140k-faces-aligned/kaggle/working/140k_aligned_faces")
MASK_ROOT = Path("/kaggle/working/140k_semantic_masks")

splits = ["train", "valid", "test"]
classes = ["real", "fake"]

MASK_ROOT.mkdir(parents=True, exist_ok=True)
print("IMG_ROOT:", IMG_ROOT)
print("MASK_ROOT:", MASK_ROOT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.3 MB/s eta 0:00:00


In [3]:
mp_face_mesh = mp.solutions.face_mesh

_FACE_MESH = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

test/fake: 100%|██████████| 10000/10000 [06:52<00:00, 24.26it/s]


In [4]:
def get_face_mesh_points(img01):
    """
    img01: [H,W,3] float in [0,1]
    returns np.array [N,2] in pixel coords, or None
    """
    H, W = img01.shape[:2]
    img_u8 = (img01 * 255).astype(np.uint8)
    res = _FACE_MESH.process(img_u8)

    if not res.multi_face_landmarks:
        return None

    lm = res.multi_face_landmarks[0]
    pts = []
    for p in lm.landmark:
        x = int(np.clip(p.x * W, 0, W - 1))
        y = int(np.clip(p.y * H, 0, H - 1))
        pts.append([x, y])
    return np.array(pts, dtype=np.int32)

In [5]:
def polygon_mask_from_points(shape_hw, pts, idxs):
    H, W = shape_hw
    mask = np.zeros((H, W), dtype=np.uint8)
    poly = pts[np.array(idxs, dtype=np.int32)]
    cv2.fillConvexPoly(mask, poly, 255)
    return mask

def build_semantic_face_masks(img01):
    """
    img01: [H,W,3] float in [0,1]
    returns masks: [10,H,W] float32 in {0,1}
    regions:
      0 full_face
      1 forehead
      2 left_eye
      3 right_eye
      4 nose
      5 mouth
      6 left_cheek
      7 right_cheek
      8 chin
      9 background
    """
    H, W = img01.shape[:2]
    pts = get_face_mesh_points(img01)

    if pts is None:
        yy, xx = np.mgrid[0:H, 0:W]
        cx, cy = W / 2.0, H / 2.0

        full = ((((xx-cx)/(W*0.33))**2 + ((yy-cy)/(H*0.40))**2) <= 1.0).astype(np.float32)

        forehead = np.zeros((H, W), np.float32); forehead[:H//3, W//4:3*W//4] = 1
        left_eye = np.zeros((H, W), np.float32); left_eye[H//3:H//2, W//4:W//2] = 1
        right_eye = np.zeros((H, W), np.float32); right_eye[H//3:H//2, W//2:3*W//4] = 1
        nose = np.zeros((H, W), np.float32); nose[H//3:2*H//3, W//3:2*W//3] = 1
        mouth = np.zeros((H, W), np.float32); mouth[2*H//3:5*H//6, W//3:2*W//3] = 1
        left_cheek = np.zeros((H, W), np.float32); left_cheek[H//2:3*H//4, W//5:2*W//5] = 1
        right_cheek = np.zeros((H, W), np.float32); right_cheek[H//2:3*H//4, 3*W//5:4*W//5] = 1
        chin = np.zeros((H, W), np.float32); chin[5*H//6:H, W//3:2*W//3] = 1

        bg = 1.0 - np.clip(full, 0, 1)

        masks = np.stack(
            [full, forehead, left_eye, right_eye, nose, mouth,
             left_cheek, right_cheek, chin, bg],
            axis=0
        ).astype(np.float32)
        return masks

    face_oval = [10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,400,377,
                 152,148,176,149,150,136,172,58,132,93,234,127,162,21,54,103,67,109]
    left_eye = [33,133,160,159,158,157,173,153,144,145]
    right_eye = [362,263,387,386,385,384,398,373,374,380]
    mouth = [61,146,91,181,84,17,314,405,321,375,291]
    nose = [6,197,195,5,4,1,19,94,2,98,327]

    face = polygon_mask_from_points((H, W), pts, face_oval)
    le = polygon_mask_from_points((H, W), pts, left_eye)
    re = polygon_mask_from_points((H, W), pts, right_eye)
    mo = polygon_mask_from_points((H, W), pts, mouth)
    no = polygon_mask_from_points((H, W), pts, nose)

    face_f = (face > 0).astype(np.float32)
    le_f = (le > 0).astype(np.float32)
    re_f = (re > 0).astype(np.float32)
    mo_f = (mo > 0).astype(np.float32)
    no_f = (no > 0).astype(np.float32)

    yy, xx = np.mgrid[0:H, 0:W]
    forehead = (face_f * (yy < H * 0.35)).astype(np.float32)
    chin = (face_f * (yy > H * 0.72)).astype(np.float32)
    left_cheek = (face_f * (xx < W * 0.45) * (yy > H * 0.42) * (yy < H * 0.72)).astype(np.float32)
    right_cheek = (face_f * (xx > W * 0.55) * (yy > H * 0.42) * (yy < H * 0.72)).astype(np.float32)
    bg = 1.0 - face_f

    masks = np.stack(
        [face_f, forehead, le_f, re_f, no_f, mo_f,
         left_cheek, right_cheek, chin, bg],
        axis=0
    ).astype(np.float32)

    return np.clip(masks, 0.0, 1.0)

In [6]:
def save_mask_file(img_path: Path, out_path: Path):
    img = Image.open(img_path).convert("RGB")
    img = np.array(img).astype(np.float32) / 255.0

    masks = build_semantic_face_masks(img)   # [10,H,W]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(out_path, masks.astype(np.float32))

In [7]:
saved = 0
failed = 0
fail_examples = []

for split in splits:
    for cls in classes:
        src_dir = IMG_ROOT / split / cls
        dst_dir = MASK_ROOT / split / cls

        files = sorted([p for p in src_dir.iterdir() if p.is_file()])

        for img_path in tqdm(files, desc=f"{split}/{cls}"):
            out_path = dst_dir / (img_path.stem + ".npy")
            try:
                save_mask_file(img_path, out_path)
                saved += 1
            except Exception as e:
                failed += 1
                if len(fail_examples) < 20:
                    fail_examples.append((str(img_path), repr(e)))

print("Saved:", saved)
print("Failed:", failed)
print("Sample failures:", fail_examples[:5])

In [8]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.14 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 whic

In [9]:
for split in splits:
    for cls in classes:
        p = MASK_ROOT / split / cls
        count = len(list(p.glob("*.npy")))
        print(split, cls, count)

2026-03-30 11:56:32.343416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774871792.779218      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774871792.889743      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774871793.866384      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774871793.866425      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774871793.866427      24 computation_placer.cc:177] computation placer alr

/usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py
True


In [10]:
!zip -r /kaggle/working/140k_semantic_masks.zip /kaggle/working/140k_semantic_masks

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [11]:
!zip -r /kaggle/working/140k_semantic_masks.zip /kaggle/working/140k_semantic_masks

In [12]:
sample = next((MASK_ROOT / "train" / "fake").glob("*.npy"))
arr = np.load(sample)
print(arr.shape, arr.dtype, arr.min(), arr.max())